In [1]:
import pandas as pd
import numpy as np


df = pd.read_csv('automobile_dirty_dataset.csv')

print(f'Shape: {df.shape}')
df.head()

Shape: (1015, 31)


,VehicleID,symboling,normalized_losses,make,fuel_type,aspiration,num_of_doors,body_style,drive_wheels,engine_location,...,compression_ratio,horsepower,peak_rpm,city_mpg,highway_mpg,price,car_color,registration_year,previous_owners,dealer_service_rating
0,V0001,0,67.0,isuzu,gas,turbo,four,hardtop,fwd,front,...,10.4,67.0,4693.0,33,40,15885,White,1985,1,5.0
1,V0002,-3,112.0,mazda,gas,turbo,four,wagon,fwd,front,...,9.8,271.0,4353.0,32,35,26156,Grey,1987,5,2.3
2,V0003,-2,162.0,chevrolet,gas,std,NaN,sedan,4wd,front,...,8.2,261.0,4600.0,29,32,27376,Grey,1990,5,4.2
3,V0004,-2,143.0,honda,gas,std,two,sedan,4wd,front,...,18.0,258.0,4898.0,30,33,NaN,Blue,1985,5,2.9
4,V0005,-2,181.0,volkswagen,gas,std,four,hatchback,fwd,front,...,9.1,228.0,6252.0,-5,40,23598,Red,1982,5,2.5


In [2]:
print(df.dtypes)
print()

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_summary[missing_summary['Missing Count'] > 0])

VehicleID                 object
symboling                  int64
normalized_losses        float64
make                      object
fuel_type                 object
aspiration                object
num_of_doors              object
body_style                object
drive_wheels              object
engine_location           object
wheel_base               float64
length                   float64
width                    float64
height                   float64
curb_weight                int64
engine_type               object
num_of_cylinders          object
engine_size                int64
fuel_system               object
bore                      object
stroke                   float64
compression_ratio        float64
horsepower               float64
peak_rpm                 float64
city_mpg                   int64
highway_mpg                int64
price                     object
car_color                 object
registration_year          int64
previous_owners            int64
dealer_ser

In [3]:
print(f'Duplicate rows: {df.duplicated().sum()}')

dirty_cats = ['fuel_type', 'aspiration']
for col in dirty_cats:
    print(f'\n{col} unique values: {df[col].unique().tolist()}')

Duplicate rows: 0

fuel_type unique values: ['gas', 'diesel', 'GAS', 'd', 'g', 'DIESEL', 'Gas', 'Diesel']

aspiration unique values: ['turbo', 'std', 'Std', 'STD', 'standard', 'Turbo', 'TURBO']


In [4]:
df['price'] = df['price'].astype(str).str.replace('$', '', regex=False).str.strip()
df['price'] = pd.to_numeric(df['price'], errors='coerce')

print('Price stats before outlier removal:')
print(df['price'].describe())

Price stats before outlier removal:
count        935.000000
mean       38854.897326
std       105479.583055
min         -500.000000
25%        19514.500000
50%        24094.000000
75%        37352.000000
max      1000000.000000
Name: price, dtype: float64


In [5]:
valid_price_mask = df['price'].between(1000, 100000)
invalid_prices = df.loc[~valid_price_mask & df['price'].notna(), 'price']
print(f'Invalid price values set to NaN ({len(invalid_prices)} rows):')
print(invalid_prices.value_counts())

df.loc[~valid_price_mask, 'price'] = np.nan

Invalid price values set to NaN (20 rows):
price
 1000000.0    6
 0.0          5
 999999.0     5
-500.0        4
Name: count, dtype: int64


In [6]:
fuel_map = {
    'gas': 'gas', 'GAS': 'gas', 'Gas': 'gas', 'g': 'gas',
    'diesel': 'diesel', 'DIESEL': 'diesel', 'Diesel': 'diesel', 'd': 'diesel'
}
df['fuel_type'] = df['fuel_type'].map(fuel_map).fillna(df['fuel_type'])
print('fuel_type after cleaning:', df['fuel_type'].unique().tolist())

fuel_type after cleaning: ['gas', 'diesel']


In [7]:
aspiration_map = {
    'std': 'std', 'STD': 'std', 'Std': 'std', 'standard': 'std',
    'turbo': 'turbo', 'Turbo': 'turbo', 'TURBO': 'turbo'
}
df['aspiration'] = df['aspiration'].map(aspiration_map).fillna(df['aspiration'])
print('aspiration after cleaning:', df['aspiration'].unique().tolist())

aspiration after cleaning: ['turbo', 'std']


In [8]:
print('Horsepower before fixing:')
print(df['horsepower'].describe())


valid_hp_mask = df['horsepower'].between(45, 700)
invalid_hp = df.loc[~valid_hp_mask & df['horsepower'].notna(), 'horsepower']
print(f'\nInvalid horsepower values set to NaN ({len(invalid_hp)} rows):')
print(invalid_hp.value_counts())

df.loc[~valid_hp_mask, 'horsepower'] = np.nan

Horsepower before fixing:
count     938.000000
mean      201.928571
std       562.108272
min       -50.000000
25%       104.250000
50%       170.000000
75%       229.000000
max      9999.000000
Name: horsepower, dtype: float64

Invalid horsepower values set to NaN (20 rows):
horsepower
-50.0      7
 800.0     6
 0.0       4
 9999.0    3
Name: count, dtype: int64


In [9]:
print('Missing values before imputation:')
print(df.isnull().sum()[df.isnull().sum() > 0])

Missing values before imputation:
normalized_losses     82
num_of_doors          80
bore                  80
stroke                81
horsepower            97
peak_rpm              82
price                100
dtype: int64


In [11]:
import re


df['bore'] = df['bore'].astype(str).str.replace(r'[^\d.]', '', regex=True)
df['bore'] = pd.to_numeric(df['bore'], errors='coerce')


numeric_cols_with_nulls = ['normalized_losses', 'bore', 'stroke', 'horsepower',
                            'peak_rpm', 'price']

for col in numeric_cols_with_nulls:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f'  {col}: filled with median = {median_val:.2f}')

  normalized_losses: filled with median = 163.00
  bore: filled with median = 3.24
  stroke: filled with median = 3.17
  horsepower: filled with median = 170.00
  peak_rpm: filled with median = 5389.00
  price: filled with median = 24068.00


In [12]:
cat_cols_with_nulls = ['num_of_doors']

for col in cat_cols_with_nulls:
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)
    print(f'  {col}: filled with mode = "{mode_val}"')

  num_of_doors: filled with mode = "four"


In [13]:
remaining_nulls = df.isnull().sum().sum()
print(f'Remaining missing values: {remaining_nulls}')

Remaining missing values: 0


In [14]:
print(f'Shape: {df.shape}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Missing values: {df.isnull().sum().sum()}')
print()
print(df[['normalized_losses', 'bore', 'stroke', 'horsepower',
          'peak_rpm', 'price']].describe().round(2))

Shape: (1015, 31)
Duplicates: 0
Missing values: 0

       normalized_losses     bore   stroke  horsepower  peak_rpm     price
count            1015.00  1015.00  1015.00     1015.00   1015.00   1015.00
mean              161.33     3.24     3.15      168.92   5373.63  27328.21
std                52.42     0.39     0.57       65.51    686.61  10083.72
min                65.00     2.54     2.07       48.00   4150.00   8802.00
25%               117.00     2.90     2.70      111.50   4776.50  19989.00
50%               163.00     3.24     3.16      170.00   5389.00  24068.00
75%               205.00     3.57     3.60      222.00   5913.50  33251.00
max               256.00     3.94     4.16      288.00   6597.00  45400.00


In [15]:
for col in ['fuel_type', 'aspiration', 'num_of_doors']:
    print(f'{col}: {sorted(df[col].unique().tolist())}')

fuel_type: ['diesel', 'gas']
aspiration: ['std', 'turbo']
num_of_doors: ['four', 'two']


In [16]:
output_path = 'automobile_clean_dataset.csv'
df.to_csv(output_path, index=False)
print(f'   Cleaned dataset saved to: {output_path}')
print(f'   Rows: {df.shape[0]}  |  Columns: {df.shape[1]}')

   Cleaned dataset saved to: automobile_clean_dataset.csv
   Rows: 1015  |  Columns: 31
